In [1]:
from agents import Agent, Production, Chat, Toolkit, Prompt, Production
from pydantic import BaseModel

### **Toolkit**

Toolkits are MCP servers that are externally run. Before initializing a Toolkit object, the MCP server needs to be operational.

In [2]:
utils_toolkit = Toolkit(name = 'Utilities', url = 'http://localhost:9001/mcp')
iris_toolkit = Toolkit(name='IRIS', url = 'http://localhost:9002/mcp')


Load started on 04/08/2026 17:40:03
Loading file Agents.Message.ToolRequest.cls as udl
Compiling class Agents.Message.ToolRequest
Compiling table Agents_Message.ToolRequest
Compiling routine Agents.Message.ToolRequest.1
Load finished successfully.

Load started on 04/08/2026 17:40:03
Loading file Agents.Message.ToolResponse.cls as udl
Compiling class Agents.Message.ToolResponse
Compiling table Agents_Message.ToolResponse
Compiling routine Agents.Message.ToolResponse.1
Load finished successfully.

Load started on 04/08/2026 17:40:03
Loading file Agents.Utils.Common.cls as udl
Compiling class Agents.Utils.Common
Compiling routine Agents.Utils.Common.1
Load finished successfully.

Load started on 04/08/2026 17:40:03
Loading file Agents.Operation.ToolkitUtilities.cls as udl
Compiling class Agents.Operation.ToolkitUtilities
Compiling routine Agents.Operation.ToolkitUtilities.1
Load finished successfully.

Load started on 04/08/2026 17:40:03
Loading file Agents.Message.ToolRequest.cls as ud

### **Chat**

The Chat API can be used to persist conversations. A chat id can be used to construct a history of that Chat from IRIS instead of needing to maintain it manually. This is particularly important when Enterprise licenses for OpenAI have Zero Data Retention enabled and so OpenAI is not authorized to store the conversation on their servers, the Chat API allows for constructing the conversation from history stored in IRIS.

In [3]:
context = Chat(
    name="travel",
    messages=[
        {"role": "system", "content": "You are helpful."},
        {"role": "user", "content": "We are in Washington DC"},
        {"role": "assistant", "content": "Great, what do you want to do in DC?"}
    ]
)
context

Chat(name='travel', messages=3)

In [4]:
context.messages

[{'role': 'system', 'content': 'You are helpful.'},
 {'role': 'user', 'content': 'We are in Washington DC'},
 {'role': 'assistant', 'content': 'Great, what do you want to do in DC?'}]

In [5]:
context == Chat('travel')

True

### **Prompt**

- The Prompt API is a way to manage and version Prompts. 
- Prompts can be built at runtime using parameters. 
- Prompts Prompts versions can be fetched by a selected version. 
- Variables contained in a prompt can be queried using `get_variables()` method.

In [6]:
bond_system = Prompt(name = 'Agent007', text = 'You are {agent_name}. You always stay in character.')
bond_system.build(agent_name='James Bond')

'You are James Bond. You always stay in character.'

In [7]:
bond_system = Prompt(name = 'Agent007', text = 'Your next mission is of utmost importance, you do not have time to talk.')
bond_system

Prompt(name='Agent007', version=2, text='Your next mission is of utmost importance, you do not have time to talk.')

In [8]:
Prompt('Agent007') == bond_system

True

In [9]:
Prompt('Agent007', version=1)

Prompt(name='Agent007', version=1, text='You are {agent_name}. You always stay in character.')

In [10]:
Prompt('Agent007', version=1).get_variables()

['agent_name']

In [11]:
Prompt('Agent007').delete()
try:
    prompt = Prompt("Agent007")
except ValueError as e:
    print(e)

No prompt found for 'Agent007'


### **Agents**

Agents can be defined by a name, a description (not currently used in any way but can be leveraged in the future for expert selection), and an OpenAI model. Optionally, agents can be configured with a default structured output (modifiable at call time) and a set of toolkits the agent should have access to. These tools are advertised to the LLM specific to access the agent has at a Toolkit level (specifying individual tools inside a Toolkit is not currently supported). Agents must be added to a Production before being used.

In [12]:
molly = Agent(name='Molly', model='gpt-5')
Production('AgentSpace', [molly]).start()
molly('What are some summer hiking trails around Boston?')


Load started on 04/08/2026 17:40:05
Loading file Agents.Utils.Common.cls as udl
Compiling class Agents.Utils.Common
Compiling routine Agents.Utils.Common.1
Load finished successfully.

Load started on 04/08/2026 17:40:05
Loading file Agents.Utils.Production.cls as udl
Compiling class Agents.Utils.Production
Compiling routine Agents.Utils.Production.1
Load finished successfully.

Load started on 04/08/2026 17:40:05
Loading file Agents.Message.LLMRequest.cls as udl
Compiling class Agents.Message.LLMRequest
Compiling table Agents_Message.LLMRequest
Compiling routine Agents.Message.LLMRequest.1
Load finished successfully.

Load started on 04/08/2026 17:40:06
Loading file Agents.Message.LLMResponse.cls as udl
Compiling class Agents.Message.LLMResponse
Compiling table Agents_Message.LLMResponse
Compiling routine Agents.Message.LLMResponse.1
Load finished successfully.

Load started on 04/08/2026 17:40:06
Loading file Agents.Message.LLMOutput.cls as udl
Compiling class Agents.Message.LLMOutp

'Here are some great summer hiking options around Boston:\n\nClose-by classics (easy to moderate):\n- Middlesex Fells Reservation (Medford/Stoneham): Skyline Trail for rocky ridges and views from Wright’s Tower; Rock Circuit for scrambles.\n- Blue Hills Reservation (Milton/Quincy): Skyline Trail via Great Blue Hill; family-friendly loops near Houghton’s Pond.\n- Lynn Woods Reservation (Lynn): Stone Tower and Dungeon Rock; lots of shaded singletrack.\n- Cutler Park (Needham/Dedham): Marsh boardwalks and river views; mostly flat.\n- Arnold Arboretum & Peters Hill (Jamaica Plain): Urban greenery with skyline views.\n\nNorth Shore:\n- Ravenswood Park (Gloucester): Cool, shaded forest trails over boulders.\n- Halibut Point State Park (Rockport): Short coastal ledges with ocean views.\n- Crane Beach & Castle Neck (Ipswich): Beach-and-dune loops; check for seasonal closures.\n\nWest/MetroWest:\n- Walden Pond (Concord): Perimeter trail; link to Fairhaven Bay or Emerson–Thoreau Amble.\n- Minute

Agents can be fetched using only their name. Adding any other parameters will be treated as agent creation.

In [13]:
Agent('Molly') == molly

True

In [14]:
class AlexResponse(BaseModel):
    message: str
    reasoning: str

class MollyResponse(BaseModel):
    text: str
    reasoning: str

alex = Agent(name='Alex', 
             description='Test Agent 1', 
             system_prompt=Prompt(name='alex_system', text='You are a helpful agent'),
             model='gpt-5',
             toolkits=[utils_toolkit],
             response_format=AlexResponse)

molly = Agent(name='Molly', 
             description='Test Agent 2', 
             system_prompt=Prompt(name='molly_system', text='You are a helpful agent'),
             model='gpt-5',
             reasoning_effort='low',
             toolkits=[utils_toolkit, iris_toolkit],
             response_format=MollyResponse)


Load started on 04/08/2026 17:40:57
Loading file Agents.Message.AlexResponse.cls as udl
Compiling class Agents.Message.AlexResponse
Compiling table Agents_Message.AlexResponse
Compiling routine Agents.Message.AlexResponse.1
Load finished successfully.

Load started on 04/08/2026 17:40:58
Loading file Agents.Message.ToolRequest.cls as udl
Compiling class Agents.Message.ToolRequest
Compiling table Agents_Message.ToolRequest
Compiling routine Agents.Message.ToolRequest.1
Load finished successfully.

Load started on 04/08/2026 17:40:58
Loading file Agents.Message.ToolResponse.cls as udl
Compiling class Agents.Message.ToolResponse
Compiling table Agents_Message.ToolResponse
Compiling routine Agents.Message.ToolResponse.1
Load finished successfully.

Load started on 04/08/2026 17:40:58
Loading file Agents.Utils.Common.cls as udl
Compiling class Agents.Utils.Common
Compiling routine Agents.Utils.Common.1
Load finished successfully.

Load started on 04/08/2026 17:40:59
Loading file Agents.Ope

In [15]:
Production('AgentSpace', [molly, alex]).start()


Load started on 04/08/2026 17:41:01
Loading file Agents.Utils.Common.cls as udl
Compiling class Agents.Utils.Common
Compiling routine Agents.Utils.Common.1
Load finished successfully.

Load started on 04/08/2026 17:41:01
Loading file Agents.Utils.Production.cls as udl
Compiling class Agents.Utils.Production
Compiling routine Agents.Utils.Production.1
Load finished successfully.

Load started on 04/08/2026 17:41:01
Loading file Agents.Message.LLMRequest.cls as udl
Compiling class Agents.Message.LLMRequest
Compiling table Agents_Message.LLMRequest
Compiling routine Agents.Message.LLMRequest.1
Load finished successfully.

Load started on 04/08/2026 17:41:02
Loading file Agents.Message.LLMResponse.cls as udl
Compiling class Agents.Message.LLMResponse
Compiling table Agents_Message.LLMResponse
Compiling routine Agents.Message.LLMResponse.1
Load finished successfully.

Load started on 04/08/2026 17:41:02
Loading file Agents.Message.LLMOutput.cls as udl
Compiling class Agents.Message.LLMOutp

In [16]:
molly(message='Which tables do we have in IRIS in the Agents namespace?')

'{"text": "The following tables are in the Agents namespace:\\n- SQLUser.Agent\\n- SQLUser.AgentToolkit\\n- SQLUser.Chat\\n- SQLUser.Prompt\\n- SQLUser.TestModel\\n- SQLUser.Toolkit\\n- SQLUser.ToolUsage\\n- SQLUser.Usage", "reasoning": "Used the previously provided IRIS list_tables tool result for the Agents namespace; no additional tool call needed."}'

In [17]:
molly(message='What is the weather today?', chat=context)

'{"text": "Today in Washington DC: Cloudy, high 26\\u00b0, low 13\\u00b0.", "reasoning": "Used the provided Utilities.weather result for Washington DC."}'

In [18]:
molly(message='Recommend some good food spots for lunch', chat='travel', reasoning_effort='high')

'{"text": "Here are solid lunch picks in DC, grouped by vibe:\\n\\nQuick and casual\\n- Little Sesame (Downtown, Chinatown) - hummus bowls and pitas; fast and fresh.\\n- Shouk (Mount Vernon Triangle, Georgetown) - plant-based Israeli pitas and bowls.\\n- RASA (Navy Yard, Dupont) - Indian bowls with great vegetarian options.\\n- Falafel Inc (Georgetown) - budget-friendly falafel sandwiches and fries.\\n- Wiseguy Pizza (Chinatown, Navy Yard) - big New York-style slices.\\n- Call Your Mother (Georgetown, Capitol Hill) - bagel sandwiches; expect a line.\\n\\nSit-down staples\\n- Old Ebbitt Grill (near the White House) - classic DC saloon; oysters and crab cakes.\\n- Zaytinya (Penn Quarter) - Eastern Mediterranean mezze by Jose Andres; easy to share.\\n- Daikaya (Chinatown) - Sapporo-style ramen; cozy and quick.\\n- Hank\'s Oyster Bar (Dupont Circle, The Wharf) - seafood and lobster rolls.\\n- Farmers Fishers Bakers (Georgetown waterfront) - broad American menu; good for groups.\\n\\nDC ico

In [19]:
class Restaurant(BaseModel):
    name: str
    cuisine: str

class TasteAtlas(BaseModel):
    restaurants: list[Restaurant]
    reasoning: str

molly(message='What are some places I would like? I tend to like Italian and Asian cuisines', response_format=TasteAtlas, chat='travel')


Load started on 04/08/2026 17:43:06
Loading file Agents.Message.Restaurant.cls as udl
Compiling class Agents.Message.Restaurant
Compiling routine Agents.Message.Restaurant.1
Load finished successfully.

Load started on 04/08/2026 17:43:06
Loading file Agents.Message.TasteAtlas.cls as udl
Compiling class Agents.Message.TasteAtlas
Compiling table Agents_Message.TasteAtlas
Compiling routine Agents.Message.TasteAtlas.1
Load finished successfully.


'{"restaurants": [{"name": "Sfoglina", "cuisine": "Italian"}, {"name": "Osteria Morini", "cuisine": "Italian"}, {"name": "L\'Ardente", "cuisine": "Italian"}, {"name": "Centrolina", "cuisine": "Italian"}, {"name": "RPM Italian", "cuisine": "Italian"}, {"name": "The Red Hen", "cuisine": "Italian"}, {"name": "Daikaya", "cuisine": "Asian - Japanese (Ramen)"}, {"name": "Haikan", "cuisine": "Asian - Japanese (Ramen)"}, {"name": "Anju", "cuisine": "Asian - Korean"}, {"name": "Maketto", "cuisine": "Asian - Cambodian/Taiwanese"}, {"name": "Thip Khao", "cuisine": "Asian - Lao"}, {"name": "Sushi Taro", "cuisine": "Asian - Japanese (Sushi)"}, {"name": "Toki Underground", "cuisine": "Asian - Taiwanese/Japanese (Ramen)"}, {"name": "Tiger Fork", "cuisine": "Asian - Hong Kong"}, {"name": "Sh\\u014dt\\u014d", "cuisine": "Asian - Japanese (Izakaya/Sushi)"}], "reasoning": "Since you\'re in DC and enjoy Italian and Asian cuisines, here are well-regarded spots across both\\u2014ranging from casual to sit-d

In [20]:
Chat('travel').usage()

'{"input_tokens": 10498, "output_tokens": 15292, "output_reasoning_tokens": 12352, "total_tokens": 25790}'

In [21]:
Production('AgentSpace').usage()

{'input_tokens': 17491,
 'output_tokens': 29563,
 'output_reasoning_tokens': 22720,
 'total_tokens': 47054}

In [22]:
molly.usage()

{'input_tokens': 17491,
 'output_tokens': 29563,
 'output_reasoning_tokens': 22720,
 'total_tokens': 47054}

In [23]:
Production('AgentSpace').usage(agents=[Agent('Molly')])

{'input_tokens': 17491,
 'output_tokens': 29563,
 'output_reasoning_tokens': 22720,
 'total_tokens': 47054}

In [24]:
Production('AgentSpace').delete()

Deleted production: User.AgentSpace

Deleting class Agents.REST.Dispatch.AgentSpaceCleaned up production-owned artifacts for: AgentSpace


In [25]:
Agent('Molly').delete()
try:
    molly('Hello')
except KeyError as e:
    print(e)


Deleting class Agents.Gateway.MollyService
Deleting class Agents.Process.Molly"No Agent found for 'Molly'"
